# WEEK 3 | INTRO TO DATABASES
## ACTIVITY: SQL — CREATE, INSERT, SELECT, GROUP BY, VIEW, INDEX
Run cells ONE BY ONE, top to bottom.

### Part A & B — Setup SQLite, CREATE TABLE, and INSERT

In [ ]:
# ── CELL 1: Setup SQLite ──────────────────────────────────────
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')  # In-memory database — resets each session
cursor = conn.cursor()
print('✅ SQLite ready. Version:', sqlite3.version)

In [ ]:
# ── CELL 2: CREATE TABLES (DDL) ──────────────────────────────
cursor.executescript('''
  CREATE TABLE team (
    team_id   INTEGER PRIMARY KEY,
    team_name TEXT NOT NULL,
    city      TEXT,
    coach     TEXT
  );

  CREATE TABLE player (
    player_id   INTEGER PRIMARY KEY,
    player_name TEXT NOT NULL,
    age         INTEGER,
    role        TEXT,
    team_id     INTEGER,
    FOREIGN KEY (team_id) REFERENCES team(team_id)
  );

  CREATE TABLE match (
    match_id        INTEGER PRIMARY KEY,
    venue           TEXT,
    team1_id        INTEGER,
    team2_id        INTEGER,
    winning_team_id INTEGER,
    FOREIGN KEY (team1_id)        REFERENCES team(team_id),
    FOREIGN KEY (team2_id)        REFERENCES team(team_id),
    FOREIGN KEY (winning_team_id) REFERENCES team(team_id)
  );
''')
conn.commit()
print('✅ Tables created: team, player, match')

In [ ]:
# ── CELL 3: INSERT DATA (DML) ────────────────────────────────
cursor.executescript('''
  INSERT INTO team VALUES (1, 'Mumbai Indians',              'Mumbai',    'Mark Boucher');
  INSERT INTO team VALUES (2, 'Royal Challengers Bangalore', 'Bengaluru', 'Andy Flower');
  INSERT INTO team VALUES (3, 'Chennai Super Kings',         'Chennai',   'Stephen Fleming');

  INSERT INTO player VALUES (1, 'Rohit Sharma',      36, 'batsman',     1);
  INSERT INTO player VALUES (2, 'Jasprit Bumrah',    30, 'bowler',      1);
  INSERT INTO player VALUES (3, 'Suryakumar Yadav',  33, 'batsman',     1);
  INSERT INTO player VALUES (4, 'Hardik Pandya',     30, 'all-rounder', 1);
  INSERT INTO player VALUES (5, 'Virat Kohli',       35, 'batsman',     2);
  INSERT INTO player VALUES (6, 'Mohammed Siraj',    29, 'bowler',      2);
  INSERT INTO player VALUES (7, 'KL Rahul',          32, 'batsman',     3);
  INSERT INTO player VALUES (8, 'Ravindra Jadeja',   35, 'all-rounder', 3);
''')
conn.commit()
print('✅ Data inserted into team and player tables')

### Part C — SELECT, WHERE, and Aliases

In [ ]:
# ── CELL 4: SELECT * — View all players ──────────────────────
result = pd.read_sql_query('SELECT * FROM player;', conn)
print('All Players:')
print(result)

In [ ]:
# ── CELL 5: SELECT with WHERE — Filter rows ──────────────────
# Professor: 'select entire row with certain constraint'
young = pd.read_sql_query('SELECT * FROM player WHERE age < 31;', conn)
print(f'Players under 31: {len(young)} rows')
print(young)

In [ ]:
# ── CELL 6: SELECT specific columns — Project ────────────────
# Professor: 'instead of select star, mention the field name'
proj = pd.read_sql_query('SELECT player_name, role FROM player;', conn)
print('Projected columns (name + role only):')
print(proj)

In [ ]:
# ── CELL 7: JOIN using WHERE — table aliases ─────────────────
# Professor: 'select s.name from student s, enroll e where s.sid = e.sid'
join_q = '''
  SELECT p.player_name, p.role, t.team_name, t.city
  FROM player p, team t
  WHERE p.team_id = t.team_id;
'''
join_result = pd.read_sql_query(join_q, conn)
print('JOIN result — each player with their team:')
print(join_result)

### Part D — Aggregate Functions (COUNT, AVG, DISTINCT)

In [ ]:
# ── CELL 8: COUNT — aggregate function ──────────────────────
# Professor: 'find average salary of instructor in CS dept'
count_q = '''
  SELECT t.team_name, COUNT(p.player_id) AS player_count
  FROM player p, team t
  WHERE p.team_id = t.team_id
  GROUP BY t.team_name;
'''
print('Players per team:')
print(pd.read_sql_query(count_q, conn))

In [ ]:
# ── CELL 9: AVG — average age per team ──────────────────────
avg_q = '''
  SELECT t.team_name, ROUND(AVG(p.age), 1) AS avg_age
  FROM player p, team t
  WHERE p.team_id = t.team_id
  GROUP BY t.team_name;
'''
print('Average age per team:')
print(pd.read_sql_query(avg_q, conn))

In [ ]:
# ── CELL 10: COUNT DISTINCT ──────────────────────────────────
# Professor: 'counting same person twice won't make any sense'
distinct_q = '''
  SELECT t.team_name, COUNT(DISTINCT p.role) AS unique_roles
  FROM player p, team t
  WHERE p.team_id = t.team_id
  GROUP BY t.team_name;
'''
print('Unique roles per team:')
print(pd.read_sql_query(distinct_q, conn))

In [ ]:
# ── CELL 11: BROKEN QUERY (intentional error!) ───────────────
# Professor: 'select clause outside aggregate must appear in GROUP BY'
# This query has team_id in SELECT but NOT in GROUP BY -- will produce error
try:
    broken_q = '''
      SELECT t.team_name, t.team_id, COUNT(p.player_id)
      FROM player p, team t
      WHERE p.team_id = t.team_id
      GROUP BY t.team_name;
    '''
    print(pd.read_sql_query(broken_q, conn))
except Exception as e:
    print(f'❌ ERROR: {e}')
    print('Fix: Add team_id to GROUP BY, or remove it from SELECT')

### Part E — HAVING Clause

In [ ]:
# ── CELL 12: HAVING — filter groups ─────────────────────────
# Professor: 'having clause only with group by -- without group by: error'
having_q = '''
  SELECT t.team_name, COUNT(p.player_id) AS player_count
  FROM player p, team t
  WHERE p.team_id = t.team_id
  GROUP BY t.team_name
  HAVING COUNT(p.player_id) > 2;
'''
print('Teams with more than 2 players:')
print(pd.read_sql_query(having_q, conn))

In [ ]:
# ── CELL 13: HAVING with AVG ─────────────────────────────────
having_avg_q = '''
  SELECT t.team_name, ROUND(AVG(p.age), 1) AS avg_age
  FROM player p, team t
  WHERE p.team_id = t.team_id
  GROUP BY t.team_name
  HAVING AVG(p.age) > 31;
'''
print('Teams where average age > 31:')
print(pd.read_sql_query(having_avg_q, conn))

### Part F — Nested Queries

In [ ]:
# ── CELL 14: NESTED QUERY — subquery in FROM ─────────────────
# Professor: 'R1, R2 in FROM clause can be sub-queries'
nested_q = '''
  SELECT p.player_name, t.team_name
  FROM player p, team t
  WHERE p.team_id = t.team_id
    AND t.city = 'Mumbai';
'''
print('Players from Mumbai-based teams:')
print(pd.read_sql_query(nested_q, conn))

In [ ]:
# ── CELL 15: NESTED QUERY — subquery in WHERE ────────────────
# Find bowlers whose age is BELOW the overall average age
sub_q = '''
  SELECT player_name, age
  FROM player
  WHERE role = 'bowler'
    AND age < (SELECT AVG(age) FROM player);
'''
print('Bowlers younger than the team average age:')
print(pd.read_sql_query(sub_q, conn))
avg_age_val = pd.read_sql_query('SELECT ROUND(AVG(age),1) AS avg FROM player', conn)
print(f'Overall average age for reference: {avg_age_val["avg"][0]}')

### Part G — Views

In [ ]:
# ── CELL 16: CREATE VIEW ─────────────────────────────────────
# Professor: 'create view with some name... stored as temporary storage'
cursor.execute('''
  CREATE VIEW young_players AS
    SELECT player_name, role, age
    FROM player
    WHERE age < 32;
''')
conn.commit()
print('✅ VIEW created: young_players')

In [ ]:
# ── CELL 17: QUERY THE VIEW ──────────────────────────────────
# Same syntax as querying a table
view_result = pd.read_sql_query('SELECT * FROM young_players;', conn)
print('Querying VIEW young_players:')
print(view_result)
print(f'Rows in view: {len(view_result)}')

In [ ]:
# ── CELL 18: DROP VIEW ───────────────────────────────────────
cursor.execute('DROP VIEW young_players;')
conn.commit()
print('✅ VIEW dropped. Base table unchanged.')

In [ ]:
# ── CELL 19: CONFIRM BASE TABLE INTACT ──────────────────────
base_intact = pd.read_sql_query('SELECT * FROM player;', conn)
print(f'player table still has {len(base_intact)} rows — view drop did NOT delete base data!')
print(base_intact[['player_name', 'age']])

### Part H — Index Demo

In [ ]:
# ── CELL 20: CREATE INDEX ─────────────────────────────────────
# Professor: 'index is basically speed up access of data retrieval'
cursor.execute('CREATE INDEX idx_age ON player(age);')
conn.commit()
print('✅ Index created on player.age')

In [ ]:
# ── CELL 21: EXPLAIN QUERY PLAN ─────────────────────────────
# See how SQLite uses the index
plan = pd.read_sql_query(
    'EXPLAIN QUERY PLAN SELECT * FROM player WHERE age < 30;', conn
)
print('Query plan with index on age:')
print(plan)
print()
print('INDEX ANALOGY: Like a textbook index page — jump directly to the right row')
print('instead of reading every page (sequential scan) to find what you need.')

### Connect: Challenge

In [ ]:
# ── CELL 22: STUDENT CHALLENGE ──────────────────────────────
# 🎯 YOUR TURN! Write the 'Rahul View' from the Connect activity

# Task 1: Create a view called rahul_courses that shows:
#   - Course title (you'll need to use the 'player' and 'team' tables as analogy
#     since we don't have a courses table — adapt the challenge)
# Task 2: Show all all-rounders and their team, sorted by age ascending
# Task 3: Find the team with the highest average age

# HINT for Task 2:
# SELECT p.player_name, t.team_name, p.age
# FROM player p, team t
# WHERE p.team_id = t.team_id AND p.role = ...
# ORDER BY ...

# YOUR CODE HERE 👇
